# Data Intelligence and outlier visualisation

Target and feature intelligence plus isolation forest, LOF, robust Mahalanobis and a Self-Organising Map (all NumPy). Nothing is saved unless you set `export_dir`.

In [ ]:
!pip install -q ml-reliability

In [ ]:
import numpy as np, pandas as pd
from sklearn.datasets import load_breast_cancer  # data loader only
from ml_reliability import DataIntelligence

data = load_breast_cancer(as_frame=True)
df = data.frame.rename(columns={'target': 'y'})
rng = np.random.default_rng(0)
df.iloc[:10, :6] *= 4  # planted outliers
df['leaky_score'] = df['y'] + rng.normal(0, 0.02, len(df))  # planted leak
print(df.shape)

In [ ]:
di = DataIntelligence(mode='autonomous')  # in-memory only
report = di.analyze(df, target_column='y')
print(report.to_text())

In [ ]:
figures = di.plot()['figures']  # in memory, nothing saved
print(list(figures))
for name in ['outlier_consensus', 'som', 'association']:
    if name in figures:
        display(figures[name])

## No target, no question?
Run the engine without a target: it proposes candidate targets and a prioritised list of questions.

In [ ]:
blind = DataIntelligence().analyze(df.drop(columns=['y', 'leaky_score']))
for c in blind['candidate_targets'][:3]:
    print(c['column'], round(c['skill'], 2), c['verdict'])
for q in blind['question_plan'][:6]:
    print(f"[{q['priority']}/{q['category']}] {q['question']}\n    next: {q['next_step']}")

## Toolkit: power, construction, active learning
Is the target learnable (predictive vs explanatory vs statistical power)? Do constructed features help on rows the search never saw? Which rows should be labelled first? `row_selector=` accepts your own selector (indices only).

In [ ]:
from ml_reliability import power_analysis, FeatureConstructor, ActiveLearningStudy, describe, plot_result
X = df.drop(columns=['y', 'leaky_score']); y = df['y'].to_numpy()
print(describe(power_analysis(X, y)))
fc = FeatureConstructor(random_state=0).fit(X, y); print(describe(fc.report))
study = ActiveLearningStudy(learner='knn', n_steps=5, n_repeats=2).run(X, y); print(describe(study))
plot_result(study)

## Optional: save the report
Uncomment to write json/txt/html to a folder.

In [ ]:
# di.export('di_report', formats=('json', 'txt', 'html'))